In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

## Reading data set

In [2]:
df = pd.read_csv("../Cars_Datasets_2025.csv", encoding="latin-1")

In [3]:
df.columns = df.columns.str.strip()

## CLEANING

In [4]:
df.head()

,Company Names,Cars Names,Engines,CC/Battery Capacity,HorsePower,Total Speed,Performance(0 - 100 )KM/H,Cars Prices,Fuel Types,Seats,Torque
0,FERRARI,SF90 STRADALE,V8,3990 cc,963 hp,340 km/h,2.5 sec,"$1,100,000",plug in hyrbrid,2,800 Nm
1,ROLLS ROYCE,PHANTOM,V12,6749 cc,563 hp,250 km/h,5.3 sec,"$460,000",Petrol,5,900 Nm
2,Ford,KA+,1.2L Petrol,"1,200 cc",70-85 hp,165 km/h,10.5 sec,"$12,000-$15,000",Petrol,5,100 - 140 Nm
3,MERCEDES,GT 63 S,V8,"3,982 cc",630 hp,250 km/h,3.2 sec,"$161,000",Petrol,4,900 Nm
4,AUDI,AUDI R8 Gt,V10,"5,204 cc",602 hp,320 km/h,3.6 sec,"$253,290",Petrol,2,560 Nm


## `Company Names` column text normalization

In [5]:
# Text normalization
df["Company Names"] = df["Company Names"].str.lower().str.strip()

In [6]:
print(df["Company Names"].value_counts().sort_index().to_string())

Company Names
acura                 27
aston martin          11
audi                  21
bentley                1
bmw                   41
bugatti               10
cadillac              20
chevrolet             58
ferrari                9
ford                  54
gmc                   59
honda                  7
hyundai               23
jaguar land rover     40
jeep                  19
kia                   71
lamborghini           24
mahindra               3
maruti suzuki          3
mazda                 87
mercedes              21
mitsubishi            60
nissan               159
peugeot               54
porsche               96
rolls royce           30
tata motors           29
tesla                 10
toyota                50
volkswagen           109
volvo                 12


#### Unique company names without any variation

In [7]:
df.head()

,Company Names,Cars Names,Engines,CC/Battery Capacity,HorsePower,Total Speed,Performance(0 - 100 )KM/H,Cars Prices,Fuel Types,Seats,Torque
0,ferrari,SF90 STRADALE,V8,3990 cc,963 hp,340 km/h,2.5 sec,"$1,100,000",plug in hyrbrid,2,800 Nm
1,rolls royce,PHANTOM,V12,6749 cc,563 hp,250 km/h,5.3 sec,"$460,000",Petrol,5,900 Nm
2,ford,KA+,1.2L Petrol,"1,200 cc",70-85 hp,165 km/h,10.5 sec,"$12,000-$15,000",Petrol,5,100 - 140 Nm
3,mercedes,GT 63 S,V8,"3,982 cc",630 hp,250 km/h,3.2 sec,"$161,000",Petrol,4,900 Nm
4,audi,AUDI R8 Gt,V10,"5,204 cc",602 hp,320 km/h,3.6 sec,"$253,290",Petrol,2,560 Nm


# `Car Names` column

In [12]:
df["Cars Names"] = df["Cars Names"].str.strip().str.lower() 

In [ ]:
print(len(df["Cars Names"].value_counts().to_string()))

In [15]:
len(df["Cars Names"].value_counts())

1199

### This column has 1199 unique values, almost one per row, so encoding it directly would not help the model (it would only memorize rows). It will be dropped, or used only to extract features.

# `Engines` column

Split engine entries with multiple values into separate rows

In [ ]:
df[df["Engines"].str.contains('/')]

### Splitting `engines` and `cc/capacity` (explode)

When a cell contains multiple values separated by `/` (e.g. `1.5L Turbo / Electric`), it is split into one row per engine.

- New `_cleaned` columns are created; the original columns are left untouched.
- If a column has a single value, it is repeated to match the length.
- Null values are converted to `[None]`.

In [11]:
df["Engines_cleaned"] = df["Engines"]
df["CC/BatteryCapacity_cleaned"] = df["CC/Battery Capacity"]

cols = ["Engines_cleaned", "CC/BatteryCapacity_cleaned"]
def igualate(row):
    n = max(len(row[c]) for c in cols)
    for c in cols:
        if len(row[c]) == 1:
            row[c] = row[c] * n
    return row

for c in cols:
    df[c] = df[c].str.strip().str.split('/')
    df[c] = df[c].apply(lambda x: x if isinstance(x, list) else [None]) 

df = df.apply(igualate, axis=1)
df = df.explode(cols, ignore_index=True)

In [12]:
df["Engines_cleaned"] = df["Engines_cleaned"].str.strip()

### Creating `displacement` column
Extracting displacement in liters from the engine description using a regex (e.g. `2.0L Turbo I4` → `2.0`) and converting it to a numeric column.

In [13]:
df["displacement"] = pd.to_numeric(
    df["Engines_cleaned"].str.extract(r"(?i)(\d+\.?\d*)\s*L\b")[0]
)
df["displacement"].isna().sum()

np.int64(480)

In [ ]:
df.loc[df["displacement"].isna(), "Engines_cleaned"].value_counts().head(30)

Some engines list displacement in cc instead of liters (e.g. `660cc Inline-3`). These values are converted to liters (cc / 1000) and used to fill the missing displacement values.

In [15]:
cc = pd.to_numeric(
    df["Engines_cleaned"].str.extract(r"(?i)(\d+)\s*cc")[0]
)
df["displacement"] = df["displacement"].fillna(cc / 1000)
df["displacement"].isna().sum()

np.int64(474)

### Creating `cylinders` column

The cylinder count appears in several formats (`V8`, `I4`, `Inline-4`, `Flat-6`, `W16`, `4-Cylinder`...). Each format is extracted with its own regex, and the results are then combined into a single `cylinders` column.

In [16]:
cyl_V = pd.to_numeric(
    df["Engines_cleaned"].str.extract(r"(?i)V(\d+)")[0]
)

cyl_I = pd.to_numeric(
    df["Engines_cleaned"].str.extract(r"(?i)I(\d+)")[0]
)

cyl_inline = pd.to_numeric(
    df["Engines_cleaned"].str.extract(r"(?i)Inline\-*(\d+)")[0]
)

cyl_flat = pd.to_numeric(
    df["Engines_cleaned"].str.extract(r"(?i)Flat\-*(\d+)")[0]
)

cyl_boxer = pd.to_numeric(
    df["Engines_cleaned"].str.extract(r"(?i)Boxer\-*(\d+)")[0]
)

cyl_W = pd.to_numeric(
    df["Engines_cleaned"].str.extract(r"(?i)W\-*(\d+)")[0]
)

cyl_VR = pd.to_numeric(
    df["Engines_cleaned"].str.extract(r"(?i)VR(\d+)")[0]
)

cyl_cylinder = pd.to_numeric(
    df["Engines_cleaned"].str.extract(r"(?i)(\d+)\-*Cylinder")[0]
)

df["cylinders"] = (cyl_V.fillna(cyl_I).fillna(cyl_inline).fillna(cyl_flat)
    .fillna(cyl_boxer).fillna(cyl_W).fillna(cyl_VR).fillna(cyl_cylinder).fillna(cyl_cylinder)
)

In [17]:
df["cylinders"].isna().sum()

np.int64(267)

In [ ]:
df.loc[df["cylinders"].isna(), "Engines_cleaned"].value_counts().head(20)

Electric motors have no cylinders, so their missing values are set to `0`. The `isna()` condition keeps hybrids (e.g. `Hybrid I4 + Electric Motor`) from being overwritten, since they already have a cylinder count.

In [19]:
mask = (
    (df["Engines_cleaned"].str.lower().str.contains("electric") | df["Engines_cleaned"].str.lower().str.contains("permanent magnet")) &
    df["cylinders"].isna()
)

df.loc[mask, "cylinders"] = 0 

In [20]:
df["cylinders"].isna().sum()

np.int64(162)

In [ ]:
df.loc[df["cylinders"].isna(), "Engines_cleaned"].value_counts().head(20)

In [ ]:
df.head()

### Creating `is_turbo` column 
Before building the flag, I listed the engines that don't match `turbo`, `supercharged` or `ecoboost` to make sure no turbocharged variant was missed. Then `is_turbo` is created as a binary column (1 = forced induction, 0 = otherwise), and a random sample of flagged rows is reviewed as a sanity check.

In [ ]:
text = df["Engines_cleaned"].str.lower()
without_turbo = ~text.str.contains("turbo|supercharged|ecoboost", na=False)
print(text[without_turbo].value_counts().to_string())

In [110]:
df["is_turbo"] = text.str.contains("turbo|supercharged|ecoboost")

In [ ]:
df.loc[df["is_turbo"] == 1, "Engines_cleaned"].sample(10)

In [112]:
text = text.str.strip()
df["is_hybrid"] = text.str.contains(r"hybrid|phev|plug|\+\s*electric", na=False).astype(int)

In [ ]:
print(df.loc[df["is_hybrid"]==0, "Engines_cleaned"].value_counts().to_string())

### Creating `is_diesel` column
Flags diesel engines (1 = diesel, 0 = otherwise). The pattern `diesel?` also catches a typo in the source data (`Turbo Diese`).

In [114]:
df["is_diesel"] = text.str.contains(r"diesel?", na=False).astype(int)

In [ ]:
print(df.loc[df["is_diesel"]==0, "Engines_cleaned"].value_counts().to_string())

### Creating `is_electric` column
Flags fully electric powertrains (1 = electric, 0 = otherwise). The pattern covers `electric`, `permanent magnet` and `fuel cell` motors. Hybrids that mention an electric motor (e.g. `I4 + Electric Motor`) are excluded using `is_hybrid`, since they also have a combustion engine.

In [116]:
mask = (
    df["Engines_cleaned"].str.lower().str.strip().str.contains("electric|permanent magnet|fuel cell") &
    (df["is_hybrid"] == 0)
)

df["is_electric"] = mask.astype(int)

In [ ]:
df.loc[df["is_electric"] == 0, "Engines_cleaned"]

### Creating `is_rotatory` column
Flags rotary (Wankel) engines (1 = rotary, 0 = otherwise). The pattern covers `rotary`, `wankel` and `rotor`, the different ways these engines are named in the data.

In [118]:
df["is_rotatory"] = text.str.lower().str.strip().str.contains("rotary|wankel|rotor", na=False).astype(int)

In [ ]:
print(df.loc[df["is_rotatory"] == 0, "Engines_cleaned"].value_counts().to_string())

### Creating `electric_motors` column
Counts the electric motors of fully electric vehicles from keywords in the description (`single` = 1, `dual` = 2, `tri` = 3, `quad` = 4). Electric engines that don't state a number default to 1, and non-electric engines get 0. Word boundaries (`\b`) prevent `tri` from matching inside `electric`.

In [120]:
number_of_electric_motors = text.str.extract(r"\b(single|dual|tri|quad)\b")[0].map(
    {"single":1, "dual":2, "tri":3, "quad":4}
)

df["electric_motors"] = np.where(df["is_electric"] == 1, number_of_electric_motors.fillna(1), 0).astype(int)

In [ ]:
print(df.loc[df["electric_motors"] > 0, "Engines_cleaned"].value_counts().to_string())

In [122]:
df.loc[df["is_electric"] == 1, "displacement"] = 0

In [123]:
df["displacement"].isna().sum()

np.int64(371)